# IDX Stock Prediction 2026: full pipeline on Colab

Predicts the next-day return of **ASII**, **BBRI** and **TLKM** for every trading day in 2026, and checks whether
the predictions beat simple baselines and buy & hold.

No GPU is needed. The whole notebook runs in about 2 minutes on a normal Colab CPU runtime.
Results (metrics, figures, predictions) land in `results/`. Commit that folder back to the repo afterwards.

In [ ]:
!git clone https://github.com/haikalfr04/Stocks-Prediction.git
%cd Stocks-Prediction
!pip install -q -r requirements.txt

## 1. Download the data

Downloads daily prices since 2015 from Yahoo Finance into `data/idx_prices.xlsx`, with one sheet per stock
(Date, Open, High, Low, Close, Adj Close, Volume) and an `Info` sheet.

In [ ]:
!python -m src.download

In [ ]:
import pandas as pd
prices = pd.read_excel("data/idx_prices.xlsx", sheet_name=None, index_col=0)
display(prices["Info"])
display(prices["BBRI"].tail())

Optional: download the Excel file to open it on your computer.

In [ ]:
from google.colab import files
files.download("data/idx_prices.xlsx")

## 2. Exploratory data analysis

Summary statistics, price history, the distribution of daily returns, and their autocorrelation. Autocorrelation
close to zero means past returns say little about the next return, which already hints at how hard the task is.

In [ ]:
!python -m src.eda
from IPython.display import Image, Markdown, display
display(Markdown(open("results/eda_stats.md").read()))
for name in ["price_history", "return_distribution", "autocorrelation"]:
    display(Image(f"results/figures/{name}.png", width=800))

## 3. Walk-forward prediction for 2026

Models are trained only on data before the day they predict. The first model is trained on 2015-2025 data, predicts
the next 21 trading days, and is then retrained with those days added. This repeats until the end of the data.

Five models are compared: three baselines (random walk, historical mean, momentum), Ridge regression and LightGBM.
Each prediction is also used in a long-or-cash trading strategy with IDX fees, compared with buy & hold.

In [ ]:
!python -m src.run

In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown(open("results/comparison.md").read()))

In [ ]:
for name in ["model_comparison", "ASII_2026_predictions", "BBRI_2026_predictions", "TLKM_2026_predictions",
             "equity_curves", "monthly_accuracy", "feature_importance"]:
    display(Image(f"results/figures/{name}.png", width=800))

Daily predictions for 2026 (one row per trading day) are saved in `results/predictions/`.

In [ ]:
daily = pd.read_csv("results/predictions/BBRI_2026.csv", index_col=0, parse_dates=True)
daily[["prev_close", "actual_close", "pred_close_lightgbm", "actual_return", "pred_return_lightgbm"]].tail(10)

## 4. Predict the next trading day

LightGBM trained on all available history predicts the day after the last date in the data.
For learning purposes only. This is not investment advice.

In [ ]:
!python -m src.predict

## 5. Download the results to commit them to GitHub

In [ ]:
!zip -qr results.zip results data/idx_prices.xlsx
from google.colab import files
files.download("results.zip")